# Apply Model as Service - exercise

```In this exercise you will experience with taking a trained models, and make them into working Application with storage, backend and user interface.```

~```Gilad Royz```

```The application you are required to build is a "Search App" that support two modes of search:```

```1. search relevant images according to textual search query. for example: "cat and dog" will return images of cant and dogs in the same picture.```

```2. search relevant texts according to image search query. for example: uploading an image of cats and dogs will return texts such as "cat and dog in the shop", "cat sitting on dog's head", ...```

```For this task, all the images and texts the application will return are images and their captions from the validtion dataset of coco2014.```

# 1. Organize the Data

```We will work the dataset: the images in the dir val2014/, and the captions in resources/image_captions.json.```

```In order to do so, we first need to organize it such that the our app could use it.```

## 1.1 Prepare Storage

In [2]:
import os
import json

import pandas as pd
import numpy as np

In [3]:
images_dir = 'val2014/'
all_image_file_names = os.listdir(images_dir)

```Download "Minio" (can be found in  https://min.io/), and run a minio server```

```The following actions do with the minio API (minio package):```

``1. Create bucket named "searchapp".``

``2. Copy all the images from the folder "val2014" into a folder named "images" inside the bucket "searchapp".``

## 1.2 Prepare DB

In [4]:
with open('resources/image_captions.json') as f:
    image_captions = json.load(f)

In [5]:
for i,caption_dict in enumerate(image_captions):
    caption_dict.update({'embedding_index': i, 'bucket': 'searchapp', 'folder': 'images'})

```Download "Postgresql" (can be found in https://www.postgresql.org/), and run postgresql server.```

```Create table named "image_captions" with the following columns:```

```
id (pk)         : int

embedding_index : int

file_name       : string

caption         : string

bucket          : string

folder          : string
```

```And insert all the values from the list "image_captions" into the table (you can use psycopg2 package)```

# 2. Create Embeddings

```In this section we want to take the images and their captions, and create a technique that will help us compare how much two images are "close" to each other, and how much two captions are "close" to each other.```

```The way we do it is by using trained Nueral Networks to encode the images and captions into fixed size vectors (array of numbers).```

## 2.1 Sentance Embeddings

````Choose your favorite Sentence Embedding model. (you can read this nice blog for ideas: https://www.analyticsvidhya.com/blog/2020/08/top-4-sentence-embedding-techniques-using-python/)```

```Use the Sentence Embedder you chose to create a matrix such that the i'th row is the embedding of the caption in the list "image_captions" with embedding_index=i.```

```Make the matrix into a numpy array, and save it as "sentence_embeddings.npy" (use np.save(..) with io.BytesIo) format in the folder "embeddings" of the bucket "searchapp".```

## 2.1 Image Embeddings

````Choose your favorite Image Embedding model. (if you don't have idea, you can look imagenet models: https://paperswithcode.com/sota/image-classification-on-imagenet)```

```Use the ImageEmbedder to create a matrix such that the i'th row is the embedding of the image in the list "image_captions" with embedding_index=i.```

```Make the matrix into a numpy array, and save it as "image_embeddings.npy" (use np.save(..) with io.BytesIo) format in the folder "embeddings" of the bucket "searchapp".```

# 3. Make "Search" Functionality

```In order to create our search app, we want to perform the two operations:```
```
1. given a sentence, create its embedding vector and find the K captions "most similar" to it among the captions in our  DB, and return their images.

2. given an image, create its embedding vector and find the K images "most similar" to it among the images in our DB, and return their captions.

```

```In order to do it, we need a similarity measure between vectors.```

```The most common way to do it is by "cosine similaruty". For vector v1, and vector v2, we define "cosine similaruty" as:```

$$similarity(v1, v2) = \frac{<v1, v2>}{\lVert v1 \rVert \cdot \lVert v2 \rVert}$$

```And it is the value of``` $cos(\theta)$, ```where``` $\theta$ ```is the angle between v1 and v2. (big similarity(v1, v2) means v1 and v2 are close to each other)```

(Explanation can be found in: https://en.wikipedia.org/wiki/Cosine_similarity)

```Write a function that gets a matrix W (each row is vector) and a vector v, and return the cosin similarity between every row in the matrix to the vector v```

In [24]:
def cosine_similarity(W, v):
    
    ## complete here ##

```Write a fnction that gets a sentence (string) and return the images of the k most similar captions to it in the DB.```

```Hint: use the embedding matrices and the DB```

```Write a fnction that gets a image (PIL.Image object) and return the captions of the k most similar images to it in the DB.```

```Hint: use the embedding matrices and the DB```

# 4. SearchApp

```Congratulations! You have everything you need in order to create the search app.```

```Now we just need to do the user interface and thats it.```

```The GUI needs to be inside the jupyter notebook with Ipywidgets package (can be found in https://ipywidgets.readthedocs.io/en/latest/).```

```After The GUI is ready, you will use "voila" to show your jupyter notebbok with the widgets as a web page. (can be found in https://github.com/voila-dashboards/voila)```

Importent:

```
1. For the GUI you will create a new notebook, and write al the code for the GUI in modular and ordered way (Object Oriented, Seperation between GUI functionality and backend functionality, ...)

2. The only access you can have to the images and captions from the GUI notebook is through the DB and Minio.```

Good luck!!!

```The search page need to have 2 tabs:```

```1. search by test: the user insert text and the page return the k most relevant images.```

```2. sreach by image: the user insert image and the page return the k most relevant captions.```

```You can decide how the rest of the UI will look.```